# Data Ingestion — Amazon Product Dataset 2020 (Home & Kitchen slice)

Pipeline: **download (kagglehub) → inspect schema → clean/normalize → embed → index (Chroma) → sample hybrid queries**.

Output of this notebook is `data/processed/products.parquet` and a persistent Chroma collection at `data/chroma_db/`, which `src/ingestion/retriever.py` queries and which the `rag.search` MCP tool will call directly.

**Why Home & Kitchen and not Household Cleaning?** The actual Kaggle file (`marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv`, 10,002 rows) is a general marketplace sample dominated by Toys & Games (6,662 rows) — only 23 rows fall under `Health & Household` at all. `Home & Kitchen` (708 rows) is the closest well-populated category to the product-discovery use case; see the README's "Known data-quality limitations" section for the full rationale and for what's missing (`Brand Name`/`Ingredients` are empty across the entire raw file; there's no rating column).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src" / "ingestion"))

## 1. Download

Requires a Kaggle API token at `~/.kaggle/kaggle.json`. Downloads via `kagglehub` and stages the CSV(s) into `data/raw/`.

In [2]:
from download_data import download

staged_files = download()
staged_files

kagglehub cached dataset at: /Users/aren/.cache/kagglehub/datasets/promptcloud/amazon-product-dataset-2020/versions/1
staged: /Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/raw/marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv (19.6 MB)


[PosixPath('/Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/raw/marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv')]

## 2. Inspect schema

PromptCloud's Amazon exports vary slightly by revision — confirm real column names before trusting the alias map in `clean.py`. If a field you care about isn't picked up, add its real column name to `COLUMN_ALIASES` in `src/ingestion/clean.py`.

For this file: `Brand Name`, `Ingredients`, `Asin`, `Sku` all exist as columns but are **entirely empty** (float64/NaN) across all 10,002 rows, and there is **no rating or review-count column at all**. `clean.py` logs a warning for missing fields rather than silently dropping them.

In [3]:
from inspect_schema import inspect

inspect()


=== marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv ===
sampled shape: (500, 28)

columns + dtypes:
  'Uniq Id': str
  'Product Name': str
  'Brand Name': float64
  'Asin': float64
  'Category': str
  'Upc Ean Code': float64
  'List Price': float64
  'Selling Price': str
  'Quantity': float64
  'Model Number': str
  'About Product': str
  'Product Specification': str
  'Technical Details': str
  'Shipping Weight': str
  'Product Dimensions': str
  'Image': str
  'Variants': str
  'Sku': float64
  'Product Url': str
  'Stock': float64
  'Product Details': float64
  'Dimensions': float64
  'Color': float64
  'Ingredients': float64
  'Direction To Use': float64
  'Is Amazon Seller': str
  'Size Quantity Variant': float64
  'Product Description': float64

first row:
{'Uniq Id': '4c69b61db1fc16e7013b43fc926e502d', 'Product Name': 'DB Longboards CoreFlex Crossbow 41" Bamboo Fiberglass Longboard Complete', 'Brand Name': nan, 'Asin': nan, 'Category': 'Sports & Outdo

## 3. Clean & normalize

- Resolves target fields (`id, title, brand, category, price, rating, features, ingredients`) via alias matching.
- Parses price out of messy `$` strings.
- Filters to the `CATEGORY_TOP_LEVEL` slice (`.env`, default `Home & Kitchen`) by exact match against the top-level segment of the `|`-delimited `Category` breadcrumb — not a keyword-in-text search, since generic phrases like "easy to clean" in product descriptions make naive keyword matching pull in unrelated products.
- Extracts a `(unit_qty, unit)` from title/weight/description text and derives `price_per_unit` for fair comparisons (622/708 rows, 88%).
- Writes `data/processed/products.parquet`, keyed by a stable `doc_id` used for citations downstream.

In [4]:
from clean import clean

products = clean()
print(products.shape)
products.head()

wrote 708 products to /Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/processed/products.parquet
(708, 13)


,id,title,brand,category,price,rating,ingredients,url,unit_qty,unit,price_per_unit,features,doc_id
0,cc2083338a16c3fe2f7895289d2e98fe,"ARTSCAPE Etched Glass 24"" x 36"" Window Film, 2...",NaN,Home & Kitchen | Home Décor | Window Treatment...,12.99,None,NaN,https://www.amazon.com/ARTSCAPE-Etched-Glass-W...,10.7,lb,1.2140,Make sure this fits by entering your model num...,cc2083338a16c3fe2f7895289d2e98fe
1,39f1b8a2129315da0288cd058b6b6086,Flash Furniture 25''W x 45''L Trapezoid Red HP...,NaN,Home & Kitchen | Furniture | Kids' Furniture |...,117.26,None,NaN,https://www.amazon.com/Flash-Furniture-Trapezo...,4.0,lb,29.3150,Collaborative Trapezoid Activity Table | Table...,39f1b8a2129315da0288cd058b6b6086
2,a11d9462309527143094a0f68bce0a58,Industro Stainless Steel Hose Clamps,NaN,Home & Kitchen | Home Décor | Kids' Room Décor,34.27,None,NaN,https://www.amazon.com/Industro-Stainless-Stee...,12.8,oz,2.6773,Make sure this fits by entering your model num...,a11d9462309527143094a0f68bce0a58
3,93c659c89f9a1a81374c61c3fdf881bb,Jay Franco Disney Frozen 2 Forest Spirit Twin/...,NaN,Home & Kitchen | Bedding | Kids' Bedding | Com...,36.37,None,NaN,https://www.amazon.com/Jay-Franco-Disney-Froze...,13.4,oz,2.7142,Make sure this fits by entering your model num...,93c659c89f9a1a81374c61c3fdf881bb
4,dbf306088532d98d12f7da924cb4e87b,"Disney's Alice in Wonderland, ""The Hatter's Ma...",NaN,Home & Kitchen | Bedding | Kids' Bedding | Bla...,35.00,None,NaN,https://www.amazon.com/Disneys-Wonderland-Hatt...,13.4,oz,2.6119,Make sure this fits by entering your model num...,dbf306088532d98d12f7da924cb4e87b


In [5]:
products[["price", "rating", "unit_qty", "price_per_unit"]].describe()

,price,unit_qty,price_per_unit
count,695.000000,633.000000,622.000000
mean,86.803971,8.713640,35.549403
std,195.787064,30.964312,74.252683
min,1.190000,0.034000,0.043300
25%,12.905000,1.600000,2.986725
50%,30.330000,3.520000,8.938150
75%,73.420000,8.000000,29.259700
max,2466.520000,473.000000,760.718800


## 4. Embed + build the Chroma index

Embedding text = `title + features + ingredients`. Metadata (`price`, `rating`, `brand`, `ingredients`, `doc_id`) is stored alongside each vector so retrieval can combine semantic similarity with metadata filters (e.g. `price <= 40`). The collection uses cosine distance (`hnsw:space="cosine"`) so the returned `score` is a bounded, interpretable cosine similarity rather than raw L2 distance.

Embedding backend is swappable via `EMBEDDING_PROVIDER` in `.env` (`local` = sentence-transformers, no API key; `openai` = `text-embedding-3-small`).

In [6]:
from build_index import build_index

build_index()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding + indexing:   0%|          | 0/12 [00:00<?, ?it/s]

embedding + indexing:   8%|▊         | 1/12 [00:02<00:23,  2.15s/it]

embedding + indexing:  17%|█▋        | 2/12 [00:02<00:10,  1.09s/it]

embedding + indexing:  25%|██▌       | 3/12 [00:02<00:06,  1.32it/s]

...

embedding + indexing:  75%|███████▌  | 9/12 [00:05<00:01,  2.43it/s]

embedding + indexing:  83%|████████▎ | 10/12 [00:05<00:00,  2.48it/s]

embedding + indexing:  92%|█████████▏| 11/12 [00:05<00:00,  2.58it/s]

embedding + indexing: 100%|██████████| 12/12 [00:05<00:00,  2.02it/s]

indexed 708 products into collection 'amazon_products_home_kitchen' at /Users/aren/Desktop/UChicago Masters/Summer/ADSP_32028_final_project/data/chroma_db


## 5. Sample hybrid queries

Semantic search + metadata filter, verified against the real index.

In [7]:
from retriever import RagRetriever, build_where

retriever = RagRetriever()
hits = retriever.search(
    "cozy throw blanket for couch",
    k=5,
    where=build_where(max_price=40),
)
for hit in hits:
    print(round(hit["score"], 3), "-", hit["title"], f"(${hit['price']})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.612 - Franco Kids Bedding Super Soft Plush Microfiber Blanket, Twin/Full Size 62" x 90", Fluffy Unicorn ($16.99)
0.578 - Ben&Jonah Throw Blankets, Perfect for The Fall & Winter. Fun Designs for Kids/Teens Great Present Friends and Family! 100% Soft Polyester. Camping/Travel Size (Rainbow Unicorn) ($24.45)
0.577 - Toy Story My New Toys Woven Tapestry Throw Blanket ($33.99)
0.575 - Subrtex Throw Twin Warm Digital Printing All Season Blanket for Bed or Couch Super Soft (Boy and Puppy, 50''x60 ($28.62)
0.559 - Jay Franco Astonish Plush Throw, Medium Blanket, Spiderman Ultimate ($20.72)


In [8]:
# Query with no filter, to sanity-check pure semantic ranking
for hit in retriever.search("birthday party balloons and decorations", k=5):
    print(round(hit["score"], 3), "-", hit["title"], f"(${hit['price']})")

0.642 - Creative Balloons 12" Latex Balloons - Pack of 72 Pieces - Decorator Cherry Red ($11.81)
0.629 - Creative Balloons Celebrity 9" Latex Balloons, Decorator Royal Blue, Pack of 144 ($16.72)
0.614 - 4.5ft Jointed Happy Birthday Banner ($2.99)
0.595 - #40 Celebration Candle | White/Red | Party Supply ($4.75)
0.575 - Celebrity 24DNB 24" Latex Balloons, Navy Blue ($17.8)


## Next steps (owned by teammates)

- `rag.search` MCP tool: wrap `RagRetriever.search()` directly, expose JSON schema `{query, k, max_price?, min_rating?, brand?}` → `build_where(...)` internally.
- Planner/Answerer agents: cite results via `doc_id`. Note `brand`/`ingredients`/`rating` will be `None` for every product in this slice — don't let the Answerer fabricate values for them.
- `web.search` MCP tool + reconciliation: match live results back to these products by title similarity (brand isn't available to match on).